In [6]:
import pandas as pd
import re
import os

# 1. Dictionary struktur file Anda
gps_files = {
    "2021_10_Oktober": ["2021Oktober/Oktober2021.csv"],
    "2021_11_November": [f"2021November/November2021_part{i}.csv" for i in range(1, 8)],
    "2021_12_Desember": [f"2021Desember/Desember2021_part{i}.csv" for i in range(1, 4)],
    "2022_01_Januari": [f"2022Januari/Januari2022_part{i}.csv" for i in range(1, 3)],
    "2022_02_Februari": [f"2022Februari/Februari2022_part{i}.csv" for i in range(1, 3)],
    "2022_03_Maret": [f"2022Maret/Maret2022_part{i}.csv" for i in range(1, 3)],
    "2022_04_April": ["2022April/April2022.csv"],
    "2022_05_Mei": [f"2022Mei/Mei2022_part{i}.csv" for i in range(1, 3)],
    "2022_06_Juni": ["2022Juni/Juni2022.csv"],
}

# Base folder untuk input dan output
BASE_INPUT_DIR = './DataGPS'
BASE_OUTPUT_DIR = './csv-anomali-records-pandas'

# Pengecekan Anomali Data GPS

Sistem ini melakukan validasi otomatis pada file CSV log GPS pada level baris (*record*). Sebuah baris dikategorikan sebagai anomali jika memenuhi salah satu kriteria berikut:

* **MAID:** Kosong atau tidak sesuai standar format UUID 36-karakter.
* **Latitude:** Bukan angka, kosong, atau di luar batas wajar (**-90.0** s/d **90.0**).
* **Longitude:** Bukan angka, kosong, atau di luar batas wajar (**-180.0** s/d **180.0**).
* **Timestamp:** Kosong, bukan angka, atau bernilai 0 dan negatif.

## Penanganan Data (Isolasi)

Baris data yang anomali tidak langsung dihapus dari *raw data*. Sistem akan mengekstrak baris tersebut dan menyimpannya di folder terpisah (`csv-anomali-records-pandas/`) dengan **struktur folder dan nama file yang sama persis** seperti sumbernya. 

Skema ini bertujuan agar data rusak mudah dilacak (*traceable*) dan diaudit tanpa mengganggu keutuhan data asli.

In [ ]:
# Fungsi validasi MAID
def is_valid_maid(maid_val):
    if pd.isna(maid_val):
        return False
    uuid_pattern = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.IGNORECASE)
    return bool(uuid_pattern.match(str(maid_val)))

# Fungsi untuk memproses 1 file dan menyimpannya jika ada anomali
def cek_dan_simpan_anomali(input_file_path, output_file_path):
    print(f"Memproses: {input_file_path}")
    
    # Load data
    try:
        df = pd.read_csv(input_file_path, low_memory=False)
    except FileNotFoundError:
        print(f"  -> [SKIPPED] File tidak ditemukan.\n")
        return
    except Exception as e:
        print(f"  -> [ERROR] Terjadi kesalahan: {e}\n")
        return

    anomalies = pd.DataFrame()

    # Cek Anomali
    invalid_maid = df[~df['maid'].apply(is_valid_maid)]
    anomalies = pd.concat([anomalies, invalid_maid])

    lat_numeric = pd.to_numeric(df['latitude'], errors='coerce')
    invalid_lat = df[(lat_numeric.isna()) | (lat_numeric < -90.0) | (lat_numeric > 90.0)]
    anomalies = pd.concat([anomalies, invalid_lat])

    lon_numeric = pd.to_numeric(df['longitude'], errors='coerce')
    invalid_lon = df[(lon_numeric.isna()) | (lon_numeric < -180.0) | (lon_numeric > 180.0)]
    anomalies = pd.concat([anomalies, invalid_lon])

    ts_numeric = pd.to_numeric(df['timestamp'], errors='coerce')
    invalid_ts = df[(ts_numeric.isna()) | (ts_numeric <= 0)]
    anomalies = pd.concat([anomalies, invalid_ts])

    # Hapus duplikat index
    anomalies = anomalies[~anomalies.index.duplicated(keep='first')]

    # Simpan hasil jika ada anomali
    if not anomalies.empty:
        print(f"  -> Ditemukan {len(anomalies)} baris anomali.")
        
        # 1. Ambil path foldernya saja dari target output file (misal: ./csv-anomali.../2021Oktober)
        output_folder = os.path.dirname(output_file_path)
        
        # 2. Buat foldernya jika belum ada (exist_ok=True mencegah error jika folder sudah ada)
        os.makedirs(output_folder, exist_ok=True)
        
        # 3. Simpan file CSV ke dalam folder tersebut
        anomalies[['maid', 'latitude', 'longitude', 'timestamp']].to_csv(output_file_path, index=False)
        print(f"  -> Disimpan ke: {output_file_path}\n")
    else:
        print("  -> Data bersih, tidak ada anomali.\n")

In [8]:
print("=== MULAI PENGECEKAN ANOMALI BATCH ===\n")

for key_bulan, list_file in gps_files.items():
    for file_name in list_file:
        # Gabungkan base folder dengan nama file dari dictionary
        input_path = os.path.join(BASE_INPUT_DIR, file_name)
        output_path = os.path.join(BASE_OUTPUT_DIR, file_name)
        
        # Eksekusi fungsi
        cek_dan_simpan_anomali(input_path, output_path)

print("=== PENGECEKAN SELESAI ===")

=== MULAI PENGECEKAN ANOMALI BATCH ===

Memproses: ./DataGPS/2021Oktober/Oktober2021.csv
  -> Ditemukan 24 baris anomali.
  -> Disimpan ke: ./csv-anomali-records-pandas/2021Oktober/Oktober2021.csv

Memproses: ./DataGPS/2021November/November2021_part1.csv
  -> Ditemukan 3 baris anomali.
  -> Disimpan ke: ./csv-anomali-records-pandas/2021November/November2021_part1.csv

Memproses: ./DataGPS/2021November/November2021_part2.csv
  -> Ditemukan 5 baris anomali.
  -> Disimpan ke: ./csv-anomali-records-pandas/2021November/November2021_part2.csv

Memproses: ./DataGPS/2021November/November2021_part3.csv
  -> Ditemukan 4 baris anomali.
  -> Disimpan ke: ./csv-anomali-records-pandas/2021November/November2021_part3.csv

Memproses: ./DataGPS/2021November/November2021_part4.csv
  -> Ditemukan 6 baris anomali.
  -> Disimpan ke: ./csv-anomali-records-pandas/2021November/November2021_part4.csv

Memproses: ./DataGPS/2021November/November2021_part5.csv
  -> Ditemukan 4 baris anomali.
  -> Disimpan ke: ./c